# Final Project EDA: COVID-19 Vaccination Around the World
#### Student Name: Rashmi Kamath
#### Date: 11/08/2026

Question I am exploring: Who got vaccinated against COVID-19, how fast, and why did some countries stall?

Smaller questions for the charts:
1. Did richer countries vaccinate more, and which countries break that pattern?
2. How did our group's four countries (United States, India, Vietnam, South Korea) compare?
3. How many people from four countries got a first dose but did not finish?
4. How did boosters compare with full vaccination in four countries ?

## Import libraries

In [190]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows() # to avoid maxrowserror in altair

DataTransformerRegistry.enable('default')

## 1. Load the data

Source: Our World in Data (OWID), COVID-19 compact dataset: https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv

Published online at OurWorldInData.org: https://ourworldindata.org/coronavirus

In [191]:
url = "https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv"
covid = pd.read_csv(url, parse_dates=["date"])
covid.head()

,country,date,total_cases,new_cases,new_cases_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths,new_deaths,...,population,population_density,median_age,life_expectancy,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities,hospital_beds_per_thousand,human_development_index
0,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
1,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
2,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
3,Afghanistan,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN
4,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1964.452271,NaN,11.7,51.938343,0.35,NaN


## 2. Look at the data

In [192]:
print(covid.shape)
covid.info()

(629868, 61)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 629868 entries, 0 to 629867
Data columns (total 61 columns):
 #   Column                                      Non-Null Count   Dtype         
---  ------                                      --------------   -----         
 0   country                                     629868 non-null  object        
 1   date                                        629868 non-null  datetime64[ns]
 2   total_cases                                 617520 non-null  float64       
 3   new_cases                                   612628 non-null  float64       
 4   new_cases_smoothed                          611419 non-null  float64       
 5   total_cases_per_million                     617520 non-null  float64       
 6   new_cases_per_million                       612628 non-null  float64       
 7   new_cases_smoothed_per_million              611419 non-null  float64       
 8   total_deaths                                617520 non-null  

The data has 629,868 rows and 61 columns. One row is one location on one day.Most columns are numbers (float64). Only `country`, `code` and `continent` are text.

The columns I will use:

- `country` : object : country or group name
- `continent` : object : continent (empty for groups like "World")
- `date` : datetime : the day of the record
- `population` : float : number of people
- `gdp_per_capita` : float : GDP per person
- `people_vaccinated_per_hundred` : float : Total number of people who received at least one vaccine dose per 100 people in the total population.(%)
- `people_fully_vaccinated_per_hundred` : float : Total number of people who received all doses prescribed by the initial vaccination protocol per 100 people in the total population.(%)
- `total_boosters_per_hundred` : float : Total number of COVID-19 vaccination booster doses administered per 100 people in the total population.
- `new_people_vaccinated_smoothed_per_hundred` : float : Daily number of people receiving their first vaccine dose (7-day smoothed) per 100 people in the total population.

## 3. Unique values

In [193]:
print("Number of locations:", covid["country"].nunique())
print("Continents:", covid["continent"].unique())
print("Date range:", covid["date"].min().date(), "to", covid["date"].max().date())

# Locations with no continent are groups, not countries
covid[covid["continent"].isna()]["country"].unique()

Number of locations: 262
Continents: ['Asia' nan 'Europe' 'Africa' 'Oceania' 'North America' 'South America']
Date range: 2020-01-01 to 2026-10-18


array(['Africa', 'Asia', 'Asia excl. China', 'England',
       'England and Wales', 'Europe', 'European Union (27)',
       'High-income countries', 'Low-income countries',
       'Lower-middle-income countries', 'North America',
       'Northern Ireland', 'Oceania', 'Scotland', 'South America',
       'Summer Olympics 2020', 'Upper-middle-income countries', 'Wales',
       'Winter Olympics 2022', 'World', 'World excl. China',
       'World excl. China and South Korea',
       'World excl. China, South Korea, Japan and Singapore'],
      dtype=object)

There are 262 locations, but some of them are groups, not countries. They have no continent, for example `World`, `High-income countries` and `European Union (27)`. If I keep them with the countries, people would be counted twice. I will remove them from the country data, but I will keep the four income groups for a separate chart.

## 4. Missing values

In [194]:
vax_cols = ["people_vaccinated_per_hundred", "people_fully_vaccinated_per_hundred",
            "total_boosters_per_hundred", "new_people_vaccinated_smoothed_per_hundred"]

(covid[vax_cols].isnull().mean() * 100).round(1)

people_vaccinated_per_hundred                 87.5
people_fully_vaccinated_per_hundred           87.8
total_boosters_per_hundred                    90.7
new_people_vaccinated_smoothed_per_hundred    67.8
dtype: float64

About 88% of rows have no vaccination numbers. This makes sense for two reasons: vaccines only started in December 2020, and many countries reported their totals only once a week or once a month, not every day. So I will drop rows with no vaccination number instead of filling them in.

## 5. Describe

In [195]:
covid[vax_cols].describe().round(1)

,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,new_people_vaccinated_smoothed_per_hundred
count,79011.0,77011.0,58784.0,202840.0
mean,52.5,47.0,30.9,0.1
std,29.5,29.3,29.5,0.2
min,0.0,0.0,0.0,0.0
25%,26.8,19.0,1.0,0.0
50%,63.1,56.4,27.7,0.0
75%,76.7,73.1,53.8,0.1
max,112.1,110.2,140.5,11.7


In [196]:
covid[(covid["people_vaccinated_per_hundred"] > 100) | (covid["people_fully_vaccinated_per_hundred"] > 100)]["country"].unique()

array(['Cook Islands', 'Gibraltar', 'Nauru'], dtype=object)

In [197]:
covid[covid["total_boosters_per_hundred"] > 100]["country"].unique()

array(['Belgium', 'Chile', 'Gibraltar', 'Guernsey', 'Japan', 'Taiwan'],
      dtype=object)

- `people_vaccinated_per_hundred` and `people_fully_vaccinated_per_hundred` go up to about 112, which should not happen for a percentage. I checked, and these come from three very small places (Gibraltar, Nauru and the Cook Islands). This can happen when many people from outside get vaccinated there, or when the population number is out of date. The 1 million population filter in the next step removes them.
- `total_boosters_per_hundred` goes above 100 because it counts doses, and one person can get more than one booster.

## 6. Clean the data

1. Keep only real countries (rows with a continent).
2. Keep countries with more than 1 million people, so very small places do not create extreme values.
3. Keep only the columns I need, and drop rows with no vaccination number.
4. Make a separate table for the four World Bank income groups.

In [198]:
countries = covid[covid["continent"].notna()]

In [199]:
countries[countries["population"] <= 1_000_000]["country"].unique()

array(['American Samoa', 'Andorra', 'Anguilla', 'Antigua and Barbuda',
       'Aruba', 'Bahamas', 'Barbados', 'Belize', 'Bermuda', 'Bhutan',
       'Bonaire Sint Eustatius and Saba', 'British Virgin Islands',
       'Brunei', 'Cape Verde', 'Cayman Islands', 'Comoros',
       'Cook Islands', 'Curacao', 'Dominica', 'Falkland Islands',
       'Faroe Islands', 'Fiji', 'French Guiana', 'French Polynesia',
       'Gibraltar', 'Greenland', 'Grenada', 'Guadeloupe', 'Guam',
       'Guernsey', 'Guyana', 'Iceland', 'Isle of Man', 'Jersey',
       'Kiribati', 'Liechtenstein', 'Luxembourg', 'Macao', 'Maldives',
       'Malta', 'Marshall Islands', 'Martinique', 'Mayotte',
       'Micronesia (country)', 'Monaco', 'Montenegro', 'Montserrat',
       'Nauru', 'New Caledonia', 'Niue', 'Northern Mariana Islands',
       'Palau', 'Reunion', 'Saint Barthelemy', 'Saint Helena',
       'Saint Kitts and Nevis', 'Saint Lucia',
       'Saint Martin (French part)', 'Saint Pierre and Miquelon',
       'Saint Vince

In [200]:

countries = countries[countries["population"] > 1_000_000]

In [201]:
keep = ["country", "continent", "date", "population", "gdp_per_capita"] + vax_cols
vax = countries[keep].dropna(subset=["people_vaccinated_per_hundred"])


In [202]:
print(vax["date"].min(), vax["date"].max())
vax.groupby("country")["date"].max().describe()

2020-12-02 00:00:00 2024-08-12 00:00:00


count                              158
mean     2023-07-08 06:31:53.924050688
min                2021-09-27 00:00:00
25%                2023-03-13 06:00:00
50%                2023-08-02 12:00:00
75%                2023-12-10 00:00:00
max                2024-08-12 00:00:00
Name: date, dtype: object

In [203]:
income_groups = ["Low-income countries", "Lower-middle-income countries",
                 "Upper-middle-income countries", "High-income countries"]
income = covid[covid["country"].isin(income_groups)][["country", "date"] + vax_cols]
income = income.dropna(subset=["people_vaccinated_per_hundred"])

In [204]:
print("Countries:", vax["country"].nunique())
print("Rows:", len(vax))
print("Rows per country (median):", vax.groupby("country").size().median())
vax.head()

Countries: 158
Rows: 53899
Rows per country (median): 186.5


,country,continent,date,population,gdp_per_capita,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,new_people_vaccinated_smoothed_per_hundred
418,Afghanistan,Asia,2021-02-22,40578847.0,1964.452271,0.000000,NaN,NaN,NaN
424,Afghanistan,Asia,2021-02-28,40578847.0,1964.452271,0.020208,NaN,NaN,0.003368
440,Afghanistan,Asia,2021-03-16,40578847.0,1964.452271,0.133074,NaN,NaN,0.007054
462,Afghanistan,Asia,2021-04-07,40578847.0,1964.452271,0.295721,NaN,NaN,0.007393
477,Afghanistan,Asia,2021-04-22,40578847.0,1964.452271,0.591441,NaN,NaN,0.019715


In [205]:
vax.isnull().sum()

country                                           0
continent                                         0
date                                              0
population                                        0
gdp_per_capita                                 1466
people_vaccinated_per_hundred                     0
people_fully_vaccinated_per_hundred            2973
total_boosters_per_hundred                    16405
new_people_vaccinated_smoothed_per_hundred      146
dtype: int64

After cleaning there are 158 countries. The median country has about 186 days with a vaccination number, but some have far fewer, so dates like "the day a country reached 50%" are only approximate.

## 7. Summary table: one row per country

For each country I take the highest value reached, because these totals only go up over time.

In [206]:
summary = vax.groupby("country").agg(
    continent=("continent", "first"),
    gdp_per_capita=("gdp_per_capita", "max"),
    one_dose=("people_vaccinated_per_hundred", "max"),
    fully=("people_fully_vaccinated_per_hundred", "max"),
    boosters=("total_boosters_per_hundred", "max"),
).reset_index()

summary["did_not_finish"] = summary["one_dose"] - summary["fully"]
print(summary[["one_dose", "fully", "boosters", "did_not_finish"]].describe().round(1))
summary.sort_values("one_dose").head(10)

       one_dose  fully  boosters  did_not_finish
count     158.0  157.0     149.0           157.0
mean       59.1   53.6      33.5             5.5
std        25.2   25.5      30.9             5.8
min         0.3    0.3       0.0             0.0
25%        41.8   33.7       7.1             2.0
50%        64.6   57.6      28.0             4.1
75%        79.6   74.4      52.8             7.1
max        98.6   98.6     140.5            48.9


,country,continent,gdp_per_capita,one_dose,fully,boosters,did_not_finish
20,Burundi,Africa,1066.512939,0.277072,0.272658,0.002357,0.004414
155,Yemen,Asia,NaN,2.747574,2.111450,0.180324,0.636124
110,Papua New Guinea,Oceania,4456.230957,3.744128,3.147961,0.337963,0.596167
59,Haiti,North America,2677.893799,4.532458,3.184559,NaN,1.347899
85,Madagascar,Africa,1668.477051,8.904759,8.636944,0.555855,0.267815
29,Congo,Africa,6194.089844,11.528550,10.838571,0.000000,0.689979
50,Gabon,Africa,18986.474609,12.804443,10.592245,0.106634,2.212198
22,Cameroon,Africa,4945.956543,13.584352,11.570632,2.907997,2.013720
136,Syria,Asia,4579.439941,14.671913,10.499275,0.337817,4.172638
44,Equatorial Guinea,Africa,13786.393555,14.976527,11.867269,0.254886,3.109258


In [207]:
summary

,country,continent,gdp_per_capita,one_dose,fully,boosters,did_not_finish
0,Afghanistan,Asia,1964.452271,47.195450,45.270844,6.727495,1.924606
1,Albania,Europe,22724.292969,47.717087,45.244260,14.230054,2.472827
2,Algeria,Africa,15882.713867,17.239624,14.251447,1.265796,2.988177
3,Angola,Africa,8760.190430,46.444870,26.965265,8.606956,19.479605
4,Argentina,South America,27846.597656,91.457770,76.860214,83.054030,14.597556
...,...,...,...,...,...,...,...
153,Venezuela,South America,NaN,78.535500,50.641060,2.309225,27.894440
154,Vietnam,Asia,15457.349609,90.787600,86.236950,58.160954,4.550650
155,Yemen,Asia,NaN,2.747574,2.111450,0.180324,0.636124
156,Zambia,Africa,3742.256348,58.113445,9.242395,NaN,48.871050


In [208]:
summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         158 non-null    object 
 1   continent       158 non-null    object 
 2   gdp_per_capita  153 non-null    float64
 3   one_dose        158 non-null    float64
 4   fully           157 non-null    float64
 5   boosters        149 non-null    float64
 6   did_not_finish  157 non-null    float64
dtypes: float64(5), object(2)
memory usage: 8.8+ KB


In [209]:
summary[summary.isnull().any(axis=1)]

,country,continent,gdp_per_capita,one_dose,fully,boosters,did_not_finish
18,Bulgaria,Europe,35325.531250,31.583748,15.926223,NaN,15.657525
32,Croatia,Europe,44172.042969,59.457554,57.634790,NaN,1.822764
33,Cuba,North America,NaN,97.701120,90.902530,79.501434,6.798590
38,Djibouti,Africa,7222.549805,37.074368,35.283410,NaN,1.790958
52,Georgia,Asia,25135.783203,43.599450,33.629680,NaN,9.969770
59,Haiti,North America,2677.893799,4.532458,3.184559,NaN,1.347899
77,Kyrgyzstan,Asia,7878.449219,24.965395,20.213600,NaN,4.751795
78,Laos,Asia,8872.694336,77.902405,69.088650,NaN,8.813755
117,Romania,Europe,40845.984375,28.562483,27.855442,NaN,0.707041
130,South Sudan,Africa,NaN,39.153057,38.773117,4.655783,0.379940


In [210]:
# Highest coverage
summary.sort_values("one_dose", ascending=False).head(10)

,country,continent,gdp_per_capita,one_dose,fully,boosters,did_not_finish
116,Qatar,Asia,110135.695312,98.60714,98.607140,65.854610,0.000000
33,Cuba,North America,NaN,97.70112,90.902530,79.501434,6.798590
148,United Arab Emirates,Asia,69701.789062,97.54937,95.608130,50.172390,1.941240
101,Nicaragua,North America,7933.488770,95.15444,91.026825,42.975430,4.127615
115,Portugal,Europe,42576.812500,94.28189,85.530440,68.026070,8.751450
98,Nepal,Asia,5276.923828,93.83405,82.351290,34.572570,11.482760
124,Singapore,Asia,139592.609375,93.57716,92.899060,84.411550,0.678100
137,Taiwan,Asia,NaN,93.50615,88.783066,108.738440,4.723084
61,Hong Kong,Asia,68724.554688,92.68869,91.124664,97.631020,1.564026
26,Chile,South America,30863.421875,92.51003,90.523636,140.514920,1.986394


## 8. Chart 1: Did richer countries vaccinate more?

- Chart type: scatter plot
- Encoding: x = GDP per person (log scale), y = % with at least one dose, color = continent
- Why log scale: GDP per person goes from about \$1,000 to over \$100,000. On a normal scale, most countries would be squeezed on the left side.
- Labels: our group's four countries are labeled.

In [220]:
our_countries = ["United States", "India", "Vietnam", "South Korea"]
scatter_data = summary.dropna(subset=["gdp_per_capita"])

points = alt.Chart(scatter_data).mark_circle(size=70, opacity=0.75).encode(
    alt.X("gdp_per_capita:Q", title="GDP per person (US dollars, log scale)",
          scale=alt.Scale(type="log", domain=[800, 150000], nice=False), axis=alt.Axis(format="$~s", values=[1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000])),
    alt.Y("one_dose:Q", title="% with at least one dose", scale=alt.Scale(domain=[0, 100])),
    alt.Color("continent:N", title="Continent", scale=alt.Scale(scheme="tableau10")),
    tooltip=["country", "continent", alt.Tooltip("gdp_per_capita:Q", format="$,.0f"), alt.Tooltip("one_dose:Q", format=".1f")]
)

labels = alt.Chart(scatter_data[scatter_data["country"].isin(our_countries)]).mark_text(
    align="left", dx=7, fontSize=11, fontWeight="bold"
).encode(x="gdp_per_capita:Q", y="one_dose:Q", text="country")

chart1 = (points + labels).properties(width=600, height=380,
    title=alt.TitleParams("Richer countries vaccinated more, but with many exceptions",
                          subtitle="Each dot is one country (over 1 million people). Hover for the name."))
chart1

alt.LayerChart(...)

In [212]:

rich = scatter_data[scatter_data["gdp_per_capita"] > 30000].sort_values("one_dose")
poor = scatter_data[scatter_data["gdp_per_capita"] < 6000].sort_values("one_dose", ascending=False)
print("\nRicher countries (GDP per person over $30k) with the lowest coverage:")
print(rich[["country", "gdp_per_capita", "one_dose"]].head(5).round(0).to_string(index=False))
print("\nPoorer countries (under $6k) with the highest coverage:")
print(poor[["country", "gdp_per_capita", "one_dose"]].head(5).round(0).to_string(index=False))


Richer countries (GDP per person over $30k) with the lowest coverage:
            country  gdp_per_capita  one_dose
            Romania         40846.0      29.0
           Bulgaria         35326.0      32.0
Trinidad and Tobago         31722.0      50.0
           Slovakia         40690.0      52.0
         Kazakhstan         37851.0      54.0

Poorer countries (under $6k) with the highest coverage:
   country  gdp_per_capita  one_dose
     Nepal          5277.0      94.0
    Rwanda          3559.0      80.0
   Myanmar          5137.0      77.0
   Liberia          1693.0      73.0
Mozambique          1451.0      70.0


What I see: There is a positive relationship, but many countries do not follow it. Some well-off Eastern European countries, like Romania and Bulgaria, stayed near 30%. Some poorer countries, like Nepal and Rwanda, reached 80% or more. GDP alone cannot explain why, so this could be a good place to merge another dataset (for example, surveys about trust in vaccines).

## 9. Chart 2: How did our four countries compare over time?

- Chart type: line chart
- Encoding: x = date, y = % with at least one dose, color = country
- Labels: country names are written at the end of each line, so readers do not have to look back and forth at a legend.

In [ ]:
four = vax[vax["country"].isin(our_countries)]

lines = alt.Chart(four).mark_line(strokeWidth=2.5).encode(
    alt.X("date:T", title="Date",
          axis=alt.Axis(format="%b %Y", tickCount={"interval": "month", "step": 6}, labelAngle=0),
          scale=alt.Scale(domain=[alt.DateTime(year=2020, month=12, date=1),
                                  alt.DateTime(year=last.year, month=last.month, date=last.day)])),
    alt.Y("people_vaccinated_per_hundred:Q", title="% with at least one dose", scale=alt.Scale(domain=[0, 100])),
    alt.Color("country:N", title="Country", sort=our_countries, scale=alt.Scale(scheme="tableau10")),
    tooltip=["country", alt.Tooltip("date:T"), alt.Tooltip("people_vaccinated_per_hundred:Q", format=".1f")]
)

last_points = four.sort_values("date").groupby("country").tail(1)
end_labels = alt.Chart(last_points).mark_text(align="left", dx=5, fontSize=11, fontWeight="bold").encode(
    x="date:T", y="people_vaccinated_per_hundred:Q", text="country",
    color=alt.Color("country:N", sort=our_countries, scale=alt.Scale(scheme="tableau10"), legend=None)
)

chart2 = (lines + end_labels).properties(width=600, height=320,
    title=alt.TitleParams("The US started fastest but finished behind Vietnam and South Korea",
                          subtitle="Share of people with at least one COVID-19 vaccine dose"))
chart2

alt.LayerChart(...)

In [214]:
# Check: when did each country reach 50% and 70% with at least one dose?
for c in our_countries:
    x = four[four["country"] == c]
    reach50 = x[x["people_vaccinated_per_hundred"] >= 50]["date"].min()
    reach70 = x[x["people_vaccinated_per_hundred"] >= 70]["date"].min()
    print(c, "| 50%:", reach50.date(), "| 70%:", reach70.date(), "| final:", round(x["people_vaccinated_per_hundred"].max(), 1))

United States | 50%: 2021-05-25 | 70%: 2021-12-15 | final: 79.1
India | 50%: 2021-10-23 | 70%: 2022-04-18 | final: 72.1
Vietnam | 50%: 2021-10-22 | 70%: 2021-11-28 | final: 90.8
South Korea | 50%: 2021-08-23 | 70%: 2021-09-18 | final: 86.4


What I see: The US had the earliest start and reached 50% first, in May 2021. After that it slowed down a lot, even though vaccines were easy to get there by then. Vietnam reached 70% in November 2021, before the US did. That fits the idea that some people did not want the vaccine, but this data cannot prove *why*. India started later but kept rising through 2021. Vietnam started last but caught up very fast at the end of 2021. India's number is also lowered because OWID divides by the whole population, and India has many children who could not get the vaccine.

## 10. Chart 3: How many people got a first dose but did not finish?

- Chart type: grouped bar chart
- Encoding: y = country, x = % of people, color = one dose vs fully vaccinated
- Labels: the value is printed on each bar.

In [215]:
four_summary = summary[summary["country"].isin(our_countries)]
four_long = four_summary.melt(id_vars="country", value_vars=["one_dose", "fully"],
                              var_name="measure", value_name="percent")
four_long["measure"] = four_long["measure"].replace({"one_dose": "At least one dose", "fully": "Fully vaccinated"})

bars = alt.Chart(four_long).mark_bar().encode(
    alt.Y("country:N", title=None, sort=our_countries),
    alt.YOffset("measure:N", sort=["At least one dose", "Fully vaccinated"]),
    alt.X("percent:Q", title="% of people", scale=alt.Scale(domain=[0, 100])),
    alt.Color("measure:N", title=None, scale=alt.Scale(range=["#9ecae1", "#08519c"])),
    tooltip=["country", "measure", alt.Tooltip("percent:Q", format=".1f")]
)
values = bars.mark_text(align="left", dx=4, fontSize=10).encode(text=alt.Text("percent:Q", format=".0f"), color=alt.value("black"))

chart3 = (bars + values).properties(width=500, height=280,
    title=alt.TitleParams("The US had the biggest gap between first dose and fully vaccinated",
                          subtitle="Highest value reported for each country"))
chart3

alt.LayerChart(...)

In [216]:
four_summary[["country", "one_dose", "fully", "did_not_finish"]].round(1)

,country,one_dose,fully,did_not_finish
63,India,72.1,66.8,5.3
129,South Korea,86.4,85.6,0.8
150,United States,79.1,67.5,11.6
154,Vietnam,90.8,86.2,4.6


What I see: The gap between "one dose" and "fully vaccinated" is about 12 points in the US, much bigger than in India, Vietnam or South Korea. So even among Americans who started, many did not finish. This is one more sign that the US problem was more about people choosing not to continue than about vaccine supply. Part of the gap may be a recording issue. In December 2021 the US CDC acknowledged that some second doses and boosters had been counted as first doses when records could not be linked, which overestimates first doses (Axios, 2021)

## 11. Chart 4: How did boosters compare with full vaccination?

- Chart type: grouped bar chart
- Encoding: x = income group, y = per 100 people, color = fully vaccinated vs booster doses
- Note: boosters are counted as doses, not people, so the two bars are not exactly the same kind of measure.

In [219]:
four_boost = four_summary[["country", "fully", "boosters"]].melt(id_vars="country", var_name="measure", value_name="per_100")
four_boost["measure"] = four_boost["measure"].replace({
    "fully": "Fully vaccinated (people)",
    "boosters": "Booster doses"})

boost_bars = alt.Chart(four_boost).mark_bar().encode(
    alt.X("country:N", title=None, sort=our_countries, axis=alt.Axis(labelAngle=0)),
    alt.XOffset("measure:N", sort=["Fully vaccinated (people)", "Booster doses"]),
    alt.Y("per_100:Q", title="Per 100 people", scale=alt.Scale(domain=[0, 100])),
    alt.Color("measure:N", title=None, sort=["Fully vaccinated (people)", "Booster doses"], scale=alt.Scale(range=["#08519c", "#fd8d3c"])),
    tooltip=["country", "measure", alt.Tooltip("per_100:Q", format=".1f")]
)
boost_values = boost_bars.mark_text(dy=-6, fontSize=10).encode(text=alt.Text("per_100:Q", format=".0f"), color=alt.value("black"))

chart4 = (boost_bars + boost_values).properties(width=520, height=320,
    title=alt.TitleParams("South Korea gave the most boosters; India gave the fewest",
                          subtitle="Highest value reported for each country"))
chart4

alt.LayerChart(...)

What I see: all four countries fully vaccinated a similar or higher share than they boosted, but boosters vary much more: South Korea gave 80 booster doses per 100 people, India 16.

## 12. Summary

Patterns to investigate
1. Richer countries vaccinated more, but income is not the whole story. Some well-off countries (Romania, Bulgaria) stayed near 30%, while some poorer countries (Nepal, Rwanda) reached 80% or more.
2. The US started first but stalled. It reached 50% with one dose by May 2021 but took until December 2021 to reach 70%. Vietnam and South Korea passed 70% before it.
3. The US had the biggest gap between first dose and fully vaccinated (about 12 points vs. under 6 in the other three). Part of this may be a recording issue: the CDC said some second doses were counted as first doses (Axios, 2021).
4. Our four countries differ most in boosters. Full vaccination ranged from 67% (India, US) to 86% (Vietnam, South Korea), but booster doses ranged from 16 per 100 in India to 80 per 100 in South Korea.

Problems with this data
- Many countries only reported once a week or once a month, so exact dates are approximate.
- Countries stopped reporting at different times (half stopped before August 2023, and the last reports are from August 2024), so "highest value reported" is not from the same date for every country.
- Booster numbers count doses, not people.
- Percentages use the whole population, including children who could not get the vaccine. This makes young countries like India look lower.
- This data shows what happened, not why. It cannot prove that people refused vaccines.

Data ideas to merge
- Survey data on vaccine hesitancy (for example, the US Census Household Pulse Survey) to check why people in the US stopped.
- Data on vaccine donations or deliveries to check if poor countries were waiting for supply.
- Age group data to compare vaccination among people who were actually eligible.